# Day 11

In [1]:
import fs from 'node:fs';

In [2]:
const input = fs.readFileSync('input.txt', 'utf-8');

In [3]:
const sample = `\
aaa: you hhh
you: bbb ccc
bbb: ddd eee
ccc: ddd eee fff
ddd: ggg
eee: out
fff: out
ggg: out
hhh: ccc fff iii
iii: out`

## Part1
Each row defines a node and it's outgoing neighbors. We need to calculate the number of different paths from `you` to `out`.

In [4]:
const process = input => new Map(input.split('\n').map(row => {
  const node = row.slice(0, 3);
  const neighbors = row.slice(5).split(' ');
  return [
    node, neighbors
  ];
}));
process(sample);

Map(10) {
  "aaa" => [ "you", "hhh" ],
  "you" => [ "bbb", "ccc" ],
  "bbb" => [ "ddd", "eee" ],
  "ccc" => [ "ddd", "eee", "fff" ],
  "ddd" => [ "ggg" ],
  "eee" => [ "out" ],
  "fff" => [ "out" ],
  "ggg" => [ "out" ],
  "hhh" => [ "ccc", "fff", "iii" ],
  "iii" => [ "out" ]
}

In [5]:
const g = process(sample);

Let's try with recursion and memoization: basically number of paths from any node is the sum of the number of paths from neighbor nodes

In [35]:
function memoize(f) {
  const d = new Map();
  function memoized_f(...args) {
    const str_args = JSON.stringify(args);
    if (d.has(str_args)) {
      return d.get(str_args);
    }
    const res = f(...args);
    d.set(str_args, res);
    return res;
  }
  return memoized_f
}

In [59]:
const fib = memoize(n => n === 1 || n === 2 ? n : fib(n-1) + fib(n-2))
fib(100)

573147844013817200000

In [60]:
const n_paths = memoize((graph, node) => {
  if (node === 'out') return 1;
  return graph.get(node).map(ne => n_paths(graph, ne)).reduce((acc, x) => acc + x);
});

In [54]:
n_paths(g, 'you')

5

In [71]:
function part1(input) {
  const graph = process(input); 
  const n_paths = memoize((node) => { // maps arent stringified correctly
    if (node === 'out') return 1;
    return graph.get(node).map(ne => n_paths(ne)).reduce((acc, x) => acc + x);
  });
  return n_paths('you');
}
part1(sample);

5

In [72]:
part1(input);

746

## Part 2

Now we need to find all the paths that lead from `svr` to `out`, but we only count paths the include both `dac` and `fft` in the path. Here I can think of a couple of approaches:

- Change the recursive function to generate all the paths and then filter down to path that satisfy our condition
- We can generalize the function to count all paths from one node to another. Then count and add
    - `n_paths('svr', 'fft') * n_paths('fft', 'dac') * n_paths('dac', 'out')
    -  `n_paths('svr', 'dac') * n_paths('dac', 'fft') * n_paths('fft', 'out')

In [94]:
const sample2 = `\
svr: aaa bbb
aaa: fft
fft: ccc
bbb: tty
tty: ccc
ccc: ddd eee
ddd: hub
hub: fff
eee: dac
dac: fff
fff: ggg hhh
ggg: out
hhh: out`

In [116]:
function n_paths2(graph, st, end) {
  const f = memoize((st, end) => { // maps arent stringified correctly
      if (st === end) return 1;
      return (graph.get(st, []) ?? []).map(ne => f(ne, end)).reduce((acc, x) => acc + x, 0);
    });
  return f(st, end);
}

n_paths2(process(input), 'you' ,'out');

746

In [117]:
const graph = process(sample2);
n_paths2(graph, 'svr', 'fft')

1

In [119]:
function part2(input) {
  const graph = process(input); 
  let total = 0;
  total += n_paths2(graph, 'svr', 'fft') * n_paths2(graph, 'fft', 'dac') * n_paths2(graph, 'dac', 'out');
  total += n_paths2(graph, 'svr', 'dac') * n_paths2(graph, 'dac', 'fft') * n_paths2(graph, 'fft', 'out');
  return total;
}
part2(input);

370500293582760

Okay, cool. Good thing I didn't try to filter.